In [1]:
pip install azure-ai-ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 74.4 MB/s  0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.0.1
    Uninstalling wrapt-2.0.1:
      Successfully uninstalled wrapt-2.0.1
  Attempting uninstall: opentelemetry-api━━━━━━━━━━━━━━━━━━━━━━━━━  7/33 [strictyaml]
    Found existing installation: opentelemetry-api 1.39.1━━━━━  7/33 [strictyaml]
    Uninstalling opentelemetry-api-1.39.1:━━━━━━━━━━━━━━━━━━━━━━━━  8/33 [opentelemetry-api]
      Successfully uninstalled opentelemetry-api-1.39.1━━━━━━━  8/33 [opentelemetry-api]
  Attempting uninstall: opentelemetry-semantic-conventions━━━━━━━━  8/33 [opentelemetry-api]
    Found existing installation: opentelemetry-semantic-conventions 0.60b1/33 [opentelemetry-api]
    Uninstalling opentelemetry-semantic-conventions-0.60b1:━━━  8/33 [opentelemetry-api]
      Successfully uninstalled opentelemetry-semantic-conventions-0.60b1 8/33 [opentelemetry-api]
  Attempting uninstall: azure-storage-blob━━━━━━

In [2]:
pip show azure-ai-ml

Name: azure-ai-ml
Version: 1.32.0
Summary: Microsoft Azure Machine Learning Client Library for Python
Home-page: https://github.com/Azure/azure-sdk-for-python
Author: Microsoft Corporation
Author-email: azuresdkengsysadmins@microsoft.com
License: MIT License
Location: /anaconda/envs/jupyter_env/lib/python3.10/site-packages
Requires: azure-common, azure-core, azure-mgmt-core, azure-monitor-opentelemetry, azure-storage-blob, azure-storage-file-datalake, azure-storage-file-share, colorama, isodate, jsonschema, marshmallow, pydash, pyjwt, pyyaml, strictyaml, tqdm, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


### Creating the compute instance and cluster:

In [4]:
# Creating the compute resource that we'll use:

% az ml compute create --name MLendpointexp \
                     --size STANDARD_DS11_V2 \
                     --type ComputeInstance \
                     --resource-group ai300proj \
                     --workspace-name mlw-ai300devArthurRReis

UsageError: Line magic function `%` not found.


In [ ]:
az ml compute create --name MLendpointexp-Cluster \ 
                     --size STANDARD_DS11_V2 \                   
                     --type AmlCompute \
                     --min-instances 0 \
                     --max-instances 4 \
                     --resource-group ai300exp \
                     --workspace-name mlw-ai300devArthurRReis

### Creating the dataset

In [ ]:
ls

In [ ]:
# Create data assets (paths relative to infra/)
echo "Create training data asset:"
az ml data create --type mltable \
  --name "house-prices-data" \
  --path data/ \
  -g ai300proj \
  -w mlw-ai300devArthurRReis

az ml data create --type uri_file \
--name "house-prices-data" \
--path data/ \
-g ai300proj \
-w mlw-ai300devArthurRReis

### Starting the main code:

In [4]:
from azure.identity import DeviceCodeCredential
from azure.ai.ml import MLClient

# Use the specific tenant ID from your error log
tenant_id = "a2a239ba-1bc9-4ac2-a3ea-a256f0dd64fa"

# 1. Create the credential
credential = DeviceCodeCredential(tenant_id=tenant_id)

# 2. FORCE the login message to appear right now
print("Attempting to authenticate...")
token = credential.get_token("https://management.azure.com/.default")
print("Login successful!")

# 3. Connect to the workspace using the now-authenticated credential
try:
    ml_client = MLClient.from_config(credential=credential)
    print(f"Connected to workspace: {ml_client.workspace_name}")
except Exception as ex:
    print(f"Workspace connection failed: {ex}")

Attempting to authenticate...
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code FQ9X24AYB to authenticate.
Login successful!
Connected to workspace: mlw-ai300devarthurrsouza


In [5]:
# Get a handle to workspace
ml_client = MLClient.from_config(credential=credential)

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [6]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

# creates a dataset based on the files in the local data folder
my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml:house-prices-data:1")

## Configure automated machine learning job

Now, you're ready to configure the automated machine learning experiment.

When you run the code below, it will create an automated machine learning job that:

- Uses the compute cluster named `aml-cluster`
- Sets `Diabetic` as the target column
- Sets `accuracy` as the primary metric
- Times out after `60` minutes of total training time 
- Trains a maximum of `5` models
- No model will be trained with the `LogisticRegression` algorithm

In [26]:
from azure.ai.ml import automl

regression_job = automl.regression(
    compute="MLendpointexp-Cluster", # Ensure this cluster exists or matches your setup
    experiment_name="auto-ml-regression-test",
    training_data=my_training_data_input,
    target_column_name="price",  # <--- Update this to your target column
    primary_metric="normalized_root_mean_squared_error", # Standard for regression
    n_cross_validations=2, 
    enable_model_explainability=True
)

regression_job.set_limits(
    timeout_minutes=15, 
    trial_timeout_minutes=5, 
    max_trials=1, # Trains only one model
    enable_early_termination=True,
)

# 3. Force Linear Regression only
regression_job.set_training(
    allowed_training_algorithms=["ElasticNet"], 
    enable_onnx_compatible_models=False,
    enable_vote_ensemble=False,  # <--- Add this
    enable_stack_ensemble=False  # <--- Add this
)

In [25]:
# This will now work without the 'misspelled' error
!az ml data download --name "house-prices-data" --version "1" \
    --resource-group "ai300proj" \
    --workspace-name "mlw-ai300devarthurrsouza" \
    --download-path "./debug_data"

'download' is misspelled or not recognized by the system.

Examples from AI knowledge base:
https://aka.ms/cli_ref
Read more about the command in reference docs


## Run an automated machine learning job

OK, you're ready to go. Let's run the automated machine learning experiment.

> **Note**: This may take some time!

In [27]:
# Submit the AutoML job
returned_job = ml_client.jobs.create_or_update(
    regression_job
)  

# submit the job to the backend
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

Monitor your job at https://ml.azure.com/runs/tender_giraffe_0k0tt5tf8p?wsid=/subscriptions/9fd0207b-08b2-4e5f-a748-8de9264eaece/resourcegroups/ai300exp/workspaces/mlw-ai300devarthurrsouza&tid=a2a239ba-1bc9-4ac2-a3ea-a256f0dd64fa


### Deploying the model

In [ ]:
from azure.ai.ml.entities import Model, ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.ai.ml.constants import AssetTypes
import datetime


In [ ]:
# 1. Define the missing variable
endpoint_name = f"price-prediction-{datetime.datetime.now().strftime('%m%d%H%M')}"

# 2. Register the model (This solves the ValidationException)
# It turns the job output into a formal asset
job_name = returned_job.name # This will use 'tender_giraffe_0k0tt5tf8p' from your log
model_name = "house-price-prediction-model"

model_asset = Model(
    path=f"azureml://jobs/{job_name}/outputs/best_model",
    name=model_name,
    type=AssetTypes.MLFLOW_MODEL, # Crucial: handles scoring automatically
    description="AutoML best model for house prices"
)
registered_model = ml_client.models.create_or_update(model_asset)


In [32]:
# 3. Create the Endpoint
endpoint = ManagedOnlineEndpoint(name=endpoint_name, auth_mode="key")
print(f"Creating endpoint: {endpoint_name}...")
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# 4. Create the Deployment using the registered model object
deployment = ManagedOnlineDeployment(
    name="price-v1",
    endpoint_name=endpoint_name,
    model=registered_model,
    instance_type="Standard_DS1_v2", # Changed from DS2 (2 cores) to DS1 (1 core)
    instance_count=1
)

Creating endpoint: price-prediction-04192134...


In [33]:
print("Deploying model... (take a coffee break, this takes ~5-8 minutes)")
ml_client.online_deployments.begin_create_or_update(deployment).result()

# 5. Route Traffic
endpoint.traffic = {"price-v1": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"✅ Deployment complete! Endpoint: {endpoint_name}")

Instance type Standard_DS1_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint price-prediction-04192134 exists


..........................

While the job is running, you can monitor it in the Studio.